# Chapter 5: Basic Notions in Graph Theory

**Colab slideshow notebook**  
Based on Chapter 5 of the book "Computational Thinking for Life Scientists"
by Benny Chor and Amir Rubinstein.

## Part III: Graphs and Networks

Many biological systems are **networks**: protein–protein interactions, gene regulation, metabolism, food webs, neurons.

A **graph** is the mathematical model of a network. In this chapter we learn the basic vocabulary of graphs,
how to represent them in Python, and two topics that build on them: **random graphs** and **clustering**.

## Chapter goals

By the end of this chapter, we want to understand:

- graphs: nodes, edges, neighbours, degree; directed and weighted graphs
- paths, cycles, distance and connectivity
- Eulerian paths (and the very different Hamiltonian paths)
- special graphs: bipartite graphs, cliques, trees
- representing graphs in Python: adjacency matrix and adjacency list
- random graphs, clusters and clustering (clique cover, k-means), phylogenetic trees

## 5.1 Graphs: definitions

A graph `G = (V, E)`: a set of **nodes** (vertices) `V` and a set of **edges** `E`, each edge a pair of nodes.

Example: proteins involved in **yeast meiosis**; an edge = an interaction between two proteins.
The interactions have no direction, so this is an **undirected graph**.

- `V = {DBF20, UME6, IME1, IME2, RIM11, STH1, HOP2}` — n = 7 nodes
- `E = {(DBF20, UME6), (DBF20, IME1), (UME6, RIM11), (UME6, IME1), (UME6, IME2), (IME1, IME2), (IME1, RIM11), (IME1, STH1), (IME2, STH1)}` — m = 9 edges

In [ ]:
# Code snippet: the yeast meiosis graph as an adjacency matrix

names = ["DBF20", "UME6", "IME1", "IME2", "RIM11", "STH1", "HOP2"]
edges = [("DBF20", "UME6"), ("DBF20", "IME1"), ("UME6", "RIM11"),
         ("UME6", "IME1"),
         ("UME6", "IME2"), ("IME1", "IME2"), ("IME1", "RIM11"), ("IME1",
             "STH1"),
         ("IME2", "STH1")]

n = len(names)
# node name -> row/column number
index = {name: i for i, name in enumerate(names)}

G = [[0] * n for i in range(n)]                    # an n by n matrix of 0's
for a, b in edges:
    G[index[a]][index[b]] = 1
    # undirected: symmetric matrix
    G[index[b]][index[a]] = 1

print("n =", n, " m =", len(edges))
for name, row in zip(names, G):
    print(f"{name:6}", row)

## 5.1 Graphs: definitions

In [ ]:
# Code snippet: drawing a graph with networkx
import networkx as nx
import matplotlib.pyplot as plt

def show_graph(G_matrix, labels=None, directed=False, colors=None):
    ''' draw a graph given as an adjacency matrix '''
    g = nx.DiGraph() if directed else nx.Graph()
    n = len(G_matrix)
    g.add_nodes_from(range(n))
    for i in range(n):
        for j in range(n):
            if G_matrix[i][j] != 0:
                g.add_edge(i, j)
    plt.figure(figsize=(6, 4))
    pos = nx.circular_layout(g)
    nx.draw(g, pos, labels={i: (labels[i] if labels else i) for i in range(n)},
            with_labels=True, node_size=1600, node_color=colors or "lightblue",
            font_size=9, arrows=directed)
    plt.show()

show_graph(G, names)

## Neighbours and degree

- Two nodes connected by an edge are **adjacent** (neighbours).
- The **neighbourhood** of a node is the set of its neighbours.
- The **degree** `deg(v)` is the number of neighbours of `v`.

In the matrix, the neighbours of node `i` are the columns `j` with `G[i][j] == 1`.

In [ ]:
# Code snippet: neighbours and degree

def neighbours(G, i):
    return [j for j in range(len(G)) if G[i][j] == 1]

def degree(G, i):
    return sum(G[i])

for name in ["IME2", "HOP2"]:
    i = index[name]
    print(f"neighbours of {name}:", [names[j] for j in neighbours(G, i)],
          "  degree:", degree(G, i))

## Neighbours and degree
### What will be the output?

In [ ]:
G = [[0, 1, 1, 0],
     [1, 0, 1, 1],
     [1, 1, 0, 0],
     [0, 1, 0, 0]]
print([sum(row) for row in G])
print([j for j in range(4) if G[1][j] == 1])

## Answer
Why: the degree of node i is the sum of row i; the neighbours of 1 are the columns with 1 in row 1.

In [ ]:
G = [[0, 1, 1, 0],
     [1, 0, 1, 1],
     [1, 1, 0, 0],
     [0, 1, 0, 0]]
print([sum(row) for row in G])
print([j for j in range(4) if G[1][j] == 1])

[2, 3, 2, 1]
[0, 2, 3]


## Directed graphs, parallel edges and self-loops

- In a **directed graph** (digraph) each edge has a direction, from a **source** to a **target**.
  Gene-regulation networks are directed: a gene regulates another, not necessarily vice versa.
- Each node has an **in-degree** (incoming edges) and an **out-degree** (outgoing edges).
- **Parallel edges:** two or more edges with the same source and target.
- **Self-loop:** an edge from a node to itself — e.g. a protein that activates or inhibits its own production.
- A **simple graph** has neither parallel edges nor self-loops.

## Directed graphs
### What will be the output?

In [ ]:
D = [[0, 1, 1],
     [0, 0, 1],
     [0, 0, 0]]
out_deg = [sum(D[i]) for i in range(3)]
in_deg = [sum(D[i][j] for i in range(3)) for j in range(3)]
print(out_deg, in_deg)

## Answer
Why: out-degree = sum of a **row** (edges leaving), in-degree = sum of a **column** (edges entering).

In [ ]:
D = [[0, 1, 1],
     [0, 0, 1],
     [0, 0, 0]]
out_deg = [sum(D[i]) for i in range(3)]
in_deg = [sum(D[i][j] for i in range(3)) for j in range(3)]
print(out_deg, in_deg)

[2, 1, 0] [0, 1, 2]


## 5.2 Weighted graphs

Edges can carry **weights**: road lengths, bond strengths, channel capacities…

In **regulation networks**: positive weight = activation, negative weight = inhibition.

Example: IME1 activates IME2 (+1), IME2 inhibits IME1 (−1);
both proteins also degrade over time — modelled as self-loops with weight −1.
We will **simulate** networks like this one in Chapter 7.

In [ ]:
# Code snippet: a small weighted, directed regulation network

reg_names = ["IME1", "IME2"]
W = [[-1, +1],    # IME1 -> IME1 (degradation), IME1 -> IME2 (activation)
     [-1, -1]]    # IME2 -> IME1 (inhibition),  IME2 -> IME2 (degradation)

for i in range(2):
    for j in range(2):
        effect = "activates" if W[i][j] > 0 else "inhibits"
        print(f"{reg_names[i]} {effect} {reg_names[j]}  (weight {W[i][j]:+d})")

## 5.3 Paths and connectivity

- A **simple path**: a sequence of distinct nodes, each connected to the next.
  Example: `(DBF20, IME1, RIM11)` — an *indirect* interaction between DBF20 and RIM11.
- A **cycle**: a path that starts and ends at the same node, e.g. `(DBF20, UME6, IME1, DBF20)`.
- The **length** of a path = its number of edges. In a weighted graph, the **weight** = sum of the edge weights.
- **Distance** `dist(v1, v2)`: the minimal length of a path from v1 to v2 (∞ if there is none).
- A graph is **connected** if there is a path between every two nodes.

In [ ]:
# Code snippet: is a sequence of nodes a path in G? what is its length?

def is_path(G, path):
    ''' True if consecutive nodes in path are connected and all nodes are
        distinct '''
    distinct = len(set(path)) == len(path)
    connected = all(G[path[k]][path[k + 1]] == 1 for k in range(len(path) - 1))
    return distinct and connected

p = [index["DBF20"], index["IME1"], index["RIM11"]]
q = [index["DBF20"], index["RIM11"]]
print("(DBF20, IME1, RIM11) is a path:", is_path(G, p), "  length:",
      len(p) - 1)
print("(DBF20, RIM11) is a path:      ", is_path(G, q))
print("degree of HOP2:", degree(G, index["HOP2"]),
      "-> the graph is not connected")

## 5.3 Paths and connectivity
In the next chapter we will see **BFS**, an efficient algorithm that computes distances — and can tell
whether a graph is connected.

## 5.4 Eulerian and Hamiltonian paths

**The house of Santa Claus:** can you draw the house in one stroke, without lifting the pen and without
drawing any line twice?

In graph terms: is there a path that goes through **every edge exactly once** — a **Eulerian path**?

## Euler's theorem (1736)

For a **connected** undirected graph (parallel edges allowed):

- (a) it has a **Eulerian cycle** ⇔ the degree of **every** node is even;
- (b) it has a **Eulerian path** ⇔ **exactly two** nodes have an odd degree — the path starts at one and ends at the other.

Intuition: every time a path "steps in" to a node it must also "step out" — using two edges.
Only the first and last nodes of a path can break this rule.

In [ ]:
# Code snippet: Euler's condition, from the degrees only

def euler_type(G):
    ''' G: connected undirected graph as a matrix; G[i][j] = number of
        edges between i and j '''
    odd = [i for i in range(len(G)) if sum(G[i]) % 2 == 1]
    if len(odd) == 0:
        return "Eulerian cycle"
    if len(odd) == 2:
        return f"Eulerian path between nodes {odd[0]} and {odd[1]}"
    return f"neither ({len(odd)} nodes of odd degree)"

# House of Santa Claus: 0 bottom-left, 1 bottom-right, 2 top-right, 3 top-left,
# 4 roof
house = [[0, 1, 1, 1, 0],
         [1, 0, 1, 1, 0],
         [1, 1, 0, 1, 1],
         [1, 1, 1, 0, 1],
         [0, 0, 1, 1, 0]]

# Seven bridges of Königsberg: 0 north, 1 south, 2 middle island, 3 east
# (parallel edges = counts)
konigsberg = [[0, 0, 2, 1],
              [0, 0, 2, 1],
              [2, 2, 0, 1],
              [1, 1, 1, 0]]

print("house of Santa Claus:", euler_type(house))
print("Königsberg bridges:  ", euler_type(konigsberg))

## Euler's theorem
### What will be the output?

In [ ]:
G = [[0, 1, 1, 0],
     [1, 0, 1, 1],
     [1, 1, 0, 1],
     [0, 1, 1, 0]]
odd = [i for i in range(4) if sum(G[i]) % 2 == 1]
print(odd)

## Answer
Why: degrees are 2, 3, 3, 2 — exactly two odd nodes, so there is a Eulerian **path** between nodes 1 and 2.

In [ ]:
G = [[0, 1, 1, 0],
     [1, 0, 1, 1],
     [1, 1, 0, 1],
     [0, 1, 1, 0]]
odd = [i for i in range(4) if sum(G[i]) % 2 == 1]
print(odd)

[1, 2]


## Exercise 1

(a) Explain why the house of Santa Claus has a Eulerian path but not a Eulerian cycle.

(b) Add one edge to the `house` matrix so that it has a Eulerian cycle, and check with `euler_type`.

**Hamiltonian path:** a path that visits every **node** exactly once.
It looks like a small change — but no efficient algorithm is known for it.
We will meet it again in Chapter 11 (*Mission Infeasible*). Same, same, but different!

## 5.5 Special types of graphs

- **Bipartite graph:** the nodes split into two sets, and every edge crosses between them
  (enzyme–substrate, gene–disease, transcription factor–target gene).
- **Clique** (fully connected graph): an edge between every pair of nodes — the densest possible sub-network.
- **Tree:** connected and acyclic (no cycles). Removing any edge disconnects it; adding any edge creates a cycle.
  A **rooted tree** has a designated root; nodes without children are **leaves**.

In [ ]:
# Code snippet: a clique on n nodes

def clique(n):
    return [[0 if i == j else 1 for j in range(n)] for i in range(n)]

for k in range(1, 6):
    C = clique(k)
    # each edge appears twice in the matrix
    m = sum(sum(row) for row in C) // 2
    print(f"clique with {k} nodes has {m} edges")

show_graph(clique(5))

## Cliques
### What will be the output?

In [ ]:
for n in [3, 4, 10]:
    print(n, n * (n - 1) // 2)

## Answer
Why: in a clique every pair of nodes is connected: n(n−1)/2 edges.

In [ ]:
for n in [3, 4, 10]:
    print(n, n * (n - 1) // 2)

3 3
4 6
10 45


## Exercises 2 and 3

**Exercise 2.** (a more difficult challenge) Find a condition that is both **necessary and sufficient** for a graph to be bipartite.
*Hint:* look at the cycles in the graph. (We will meet an efficient test in Chapter 11.)

**Exercise 3.** How many edges does a clique with `n` nodes have
(a) without self-loops? (b) with self-loops? Check your formula against the output above.

## 5.6 Representing graphs

Two common representations, shown for an undirected graph with nodes `0, 1, 2, 3`
(edges 0–1, 0–2, 0–3, 1–2):

- **Adjacency matrix:** an `n × n` matrix; `G[i][j] == 1` iff there is an edge between i and j.
  For an undirected graph it is **symmetric**.
- **Adjacency list:** a list of lists; the i-th inner list holds the neighbours of node i.

In [ ]:
# Code snippet: the same graph in both representations, and converting between
# them

G_matrix = [[0, 1, 1, 1],
            [1, 0, 1, 0],
            [1, 1, 0, 0],
            [1, 0, 0, 0]]

def matrix_to_list(M):
    return [[j for j in range(len(M)) if M[i][j] == 1] for i in range(len(M))]

def list_to_matrix(L):
    n = len(L)
    M = [[0] * n for i in range(n)]
    for i in range(n):
        for j in L[i]:
            M[i][j] = 1
    return M

G_list = matrix_to_list(G_matrix)
print("adjacency list:", G_list)
print("back to a matrix equals the original:",
      list_to_matrix(G_list) == G_matrix)
print("G_matrix[1][2] =", G_matrix[1][2], "   G_matrix[3][1] =",
      G_matrix[3][1])

## 5.6 Representing graphs
### What will be the output?

In [ ]:
M = [[0, 1, 0],
     [1, 0, 1],
     [0, 1, 0]]
print([[j for j in range(3) if M[i][j] == 1] for i in range(3)])

## Answer
Why: the adjacency list keeps, for each node, only the columns where its row has a 1.

In [ ]:
M = [[0, 1, 0],
     [1, 0, 1],
     [0, 1, 0]]
print([[j for j in range(3) if M[i][j] == 1] for i in range(3)])

[[1], [0, 2], [1]]


## Exercises 4 and 5

**Exercise 4.** In a **directed** graph, what does the i-th **row** of the adjacency matrix represent? And the j-th **column**?

**Exercise 5.** When would you prefer each representation? Consider:

(a) how to include edge weights in each,
(b) memory usage,
(c) checking whether two given nodes are connected,
(d) printing all neighbours of a given node.

## 5.7 Random graphs

Random graphs are generated by a probabilistic process. They are used to model phase transitions, neurons,
traffic, social acquaintances — and as a **baseline**: a structure that appears in a biological network far more
often than in random graphs hints at a biological role preserved by evolution.

**Erdős–Rényi model (1959):** n nodes; connect every pair independently with probability `p`.

**Exercise 6.** Complete `rand_graph(n, p)`, which returns a random undirected graph as a matrix.
(`random.random()` returns a random number between 0 and 1.) Then compare with the solution below.

In [ ]:
# Code snippet: solution to Exercise 6 — an Erdős–Rényi random graph
import random

def rand_graph(n, p):
    G = []
    for i in range(n):
        G.append([0] * n)

    for i in range(n):
        for j in range(i + 1, n):
            r = random.random()
            if r < p:
                G[i][j] = 1
                G[j][i] = 1
    return G

random.seed(0)
for p in [0.1, 0.3, 0.6]:
    R = rand_graph(10, p)
    print(f"p = {p}: number of edges =", sum(sum(row) for row in R) // 2)
    show_graph(R)

## Preferential attachment: "the rich get richer"

- Nodes with a **higher degree** are **more likely** to get new edges.
- Defined by Yule (1925) for the emergence of species; studied by Barabási and Albert (1999).
- Claimed to resemble biological networks and the internet better than Erdős–Rényi: a few **hubs** with very high degree.

In [ ]:
# Code snippet: degrees in an Erdős–Rényi graph vs. a preferential-attachment
# graph (same number of edges)

# random graph with 200 nodes, 396 edges
er = nx.gnm_random_graph(200, 396, seed=1)
# preferential attachment, 200 nodes, 396 edges
ba = nx.barabasi_albert_graph(200, 2, seed=1)

print("edges:", er.number_of_edges(), ba.number_of_edges())
print("maximal degree, Erdős–Rényi:          ", max(d for _, d in er.degree()))
print("maximal degree, preferential attachment:",
      max(d for _, d in ba.degree()))

plt.figure(figsize=(8, 3))
plt.hist([d for _, d in er.degree()], bins=range(0, 40), alpha=0.6,
         label="Erdős–Rényi")
plt.hist([d for _, d in ba.degree()], bins=range(0, 40), alpha=0.6,
         label="preferential attachment")
plt.xlabel("degree"); plt.ylabel("number of nodes"); plt.legend(); plt.show()

## 5.8 Clusters and clustering

**Clustering:** partition objects into sets (**clusters**) so that objects in the same cluster are similar,
and objects in different clusters are less similar.

- Genes are often clustered by their expression levels in different conditions.
- **"Guilt by association":** an unknown gene in a cluster of genes with a known function may share that function.
- In social networks, clusters are called **communities**.

When we know which pairs of objects are related, we get a graph: an edge = "similar".

## Clique cover

A natural idea: partition the nodes into **cliques** — every node in exactly one clique (a **clique cover**).

- Goal: as **few** cliques as possible. Finding the minimum is a computationally **hard** problem
  (only exponential-time algorithms are known — see Chapter 11).
- So we use a **heuristic**: fast, and usually good, but not guaranteed optimal.

**Greedy clique-cover heuristic:**
1. Start with every node as its own cluster.
2. Repeat until no two clusters can be merged: for each cluster C, among the other clusters that are
   **fully connected** to C, pick the largest one D, and merge C with D.

In [ ]:
# Code snippet: the greedy clique-cover heuristic

def fully_connected(G, A, B):
    return all(G[a][b] == 1 for a in A for b in B)

def greedy_clique_cover(G, order):
    ''' order: the order in which clusters are visited (a list of the
        nodes) '''
    clusters = [[v] for v in order]
    merged = True
    while merged:
        merged = False
        for C in clusters:
            candidates = [D for D in clusters if D is not C
                          and fully_connected(G, C, D)]
            if candidates:
                D = max(candidates, key=len)
                clusters.remove(D)
                C.extend(D)
                merged = True
                break
    return clusters

# an example graph: A,B,C form a triangle, D,E,F form a triangle, plus edges
# B-D and C-D
labels = ["A", "B", "C", "D", "E", "F"]
H = [[0] * 6 for _ in range(6)]
for a, b in [("A","B"), ("A","C"), ("B","C"), ("D","E"), ("D","F"),
             ("E","F"), ("B","D"), ("C","D")]:
    i, j = labels.index(a), labels.index(b)
    H[i][j] = H[j][i] = 1

for order in [[0, 1, 2, 3, 4, 5], [1, 3, 0, 2, 4, 5]]:
    cover = greedy_clique_cover(H, order)
    print("visiting order", [labels[v] for v in order], "->",
          [[labels[v] for v in C] for C in cover])
show_graph(H, labels)

## Greedy algorithms

The heuristic is **greedy** (myopic): in each step it makes the locally best choice, ignoring the "bigger picture".

- Simple and efficient
- Sometimes optimal — but often not: as above, a different visiting order can give **more** cliques

## Exercise 7

(a) Explain, step by step, why the second visiting order ended with **3** cliques instead of 2.

(b) Is there a visiting order for which the heuristic produces **4** cliques? Try a few orders in code.

## k-means clustering

For objects that are **points** (vectors of numbers), with a distance between them:

1. Choose `k` of the points at random as initial **centers**.
2. Repeat until the clusters stop changing:
   - assign every point to its **closest** center;
   - move every center to the **average** ("center of mass") of its cluster.

Fast and very popular (e.g. clustering genes by expression). But the result depends on the random start,
and we must choose `k` in advance.

In [ ]:
# Code snippet: k-means on random points in the plane
import math

def kmeans(points, k, seed=0):
    random.seed(seed)
    centers = random.sample(points, k)
    clusters = None
    while True:
        new_clusters = [min(range(k), key=lambda c: math.dist(p, centers[c]))
                        for p in points]
        if new_clusters == clusters:            # no change -> stop
            return centers, clusters
        clusters = new_clusters
        for c in range(k):
            members = [p for p, cl in zip(points, clusters) if cl == c]
            if members:
                centers[c] = (sum(x for x, y in members) / len(members),
                              sum(y for x, y in members) / len(members))

random.seed(1)
points = [(random.gauss(cx, 0.6), random.gauss(cy, 0.6))
          for cx, cy in [(0, 0), (4, 1), (2, 4)] for _ in range(40)]

centers, clusters = kmeans(points, 3)
plt.figure(figsize=(5, 5))
plt.scatter([x for x, y in points], [y for x, y in points], c=clusters,
            cmap="viridis")
plt.scatter([x for x, y in centers], [y for x, y in centers], marker="s",
            s=150, c="red")
plt.title("k-means with k = 3 (red squares = centers)")
plt.show()

## 5.9 Hierarchical clustering and phylogenetic trees

- In **hierarchical** clustering an object belongs to several nested clusters: any two clusters are either
  disjoint, or one contains the other.
- This naturally forms a **rooted tree**; the root = all objects.
- **Phylogenetic trees** describe evolutionary relationships. *Homo sapiens* ⊂ Primates ⊂ Mammalia ⊂ Vertebrata ⊂ Eukarya.
- Today they are built mostly from **molecular data** (aligned DNA or protein sequences).
- Most reconstruction methods (e.g. **maximum parsimony**) optimise an objective that is computationally hard;
  heuristics usually give satisfactory trees.

## Challenge Yourself — Problem 1: graph isomorphism

Two graphs are **isomorphic** if they have the same structure: the nodes of one can be renamed
(keeping the edges) to obtain exactly the other.

(a) Find two graphs that are **not** isomorphic although they have the same distribution of node degrees.

(b) Conclude: is "same degree distribution" a **necessary** or a **sufficient** condition for isomorphism?

## Challenge Yourself — Problem 2: hubs

A **hub** is a node with an exceptionally large degree. Here: a node with the **maximal** in-degree or out-degree.

Complete `hub(G, inout)`: `G` is a directed, non-weighted graph (0/1 matrix), `inout` is `"in"` or `"out"`.
Return a pair `(maximal degree, node index)`.

In [ ]:
# Problem 2: your solution here

def hub(G, inout):
    ''' Find the node with maximal in/out degree in G.
        G is a non-weighted digraph represented as a 0/1 matrix.
        inout = "in" or "out"
        Returns a pair (maximal degree, corresponding node) '''
    max_deg = 0
    hub_node = None
    for i in range(len(G)):
        if inout == "in":
            deg = 0        # replace: the in-degree of node i
        elif inout == "out":
            deg = 0        # replace: the out-degree of node i
        else:              # inout must be "in" or "out"
            return
        if deg > max_deg:
            max_deg = deg
            hub_node = i
    return max_deg, hub_node

D = [[0, 1, 1, 1],
     [0, 0, 1, 0],
     [0, 0, 0, 0],
     [0, 1, 1, 0]]
print('hub(D, "out") =', hub(D, "out"), "  expected: (3, 0)")
print('hub(D, "in")  =', hub(D, "in"), "  expected: (3, 2)")

## Chapter 5 Summary

In this chapter we studied:

- graphs: nodes, edges, neighbours, degree; directed, weighted, simple graphs
- paths, cycles, distance and connectivity
- Euler's theorem for Eulerian paths and cycles; Hamiltonian paths (hard — Chapter 11)
- bipartite graphs, cliques, trees
- adjacency matrix vs. adjacency list
- random graphs: Erdős–Rényi and preferential attachment
- clustering: greedy clique cover, k-means, hierarchical clustering and phylogenetic trees